# Inferential Analysis of the Corrected Final Model

This notebook implements the corrected inferential specification without the dropped interaction term.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

DATA_PATH = Path.cwd().parent / 'data' / 'final' / 'final.corrected.csv'
df = pd.read_csv(DATA_PATH)
OUTCOME = 'log_box_office'
PREDICTORS = ['audienceScore', 'initial_combined_sentiment_score', 'log_initial_review_count']
FORMULA = 'log_box_office ~ audienceScore + initial_combined_sentiment_score + log_initial_review_count'

for col in [OUTCOME, *PREDICTORS, 'box_office_num']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
model_df = df[np.isfinite(df[OUTCOME])].dropna(subset=[OUTCOME, *PREDICTORS]).copy()
print(f'Data path: {DATA_PATH}')
print(f'Inferential sample: {len(model_df):,}')


## Correlation Checks

In [ ]:
correlation_results = []
for predictor in PREDICTORS:
    pair_df = model_df[[predictor, OUTCOME]].dropna()
    pearson_r, pearson_p = stats.pearsonr(pair_df[predictor], pair_df[OUTCOME])
    spearman_rho, spearman_p = stats.spearmanr(pair_df[predictor], pair_df[OUTCOME])
    correlation_results.append({
        'predictor': predictor,
        'pearson_r': pearson_r,
        'pearson_p': pearson_p,
        'spearman_rho': spearman_rho,
        'spearman_p': spearman_p,
    })
pd.DataFrame(correlation_results).round(4)

## Corrected OLS Model

In [ ]:
X = sm.add_constant(model_df[PREDICTORS])
y = model_df[OUTCOME]
ols_model = sm.OLS(y, X).fit(cov_type='HC3')
ols_model_classical = smf.ols(FORMULA, data=model_df).fit()
anova_table = sm.stats.anova_lm(ols_model_classical, typ=1)

coefficient_table = pd.DataFrame({
    'term': ols_model.params.index,
    'coef': ols_model.params.values,
    'std_err_hc3': ols_model.bse.values,
    'z_or_t': ols_model.tvalues.values,
    'p_value': ols_model.pvalues.values,
})

model_fit_summary = pd.DataFrame([{
    'n_obs': int(ols_model.nobs),
    'r_squared': float(ols_model.rsquared),
    'adj_r_squared': float(ols_model.rsquared_adj),
    'f_statistic': float(ols_model.fvalue),
    'model_p_value': float(ols_model.f_pvalue),
}])

coefficient_table.round(4), model_fit_summary.round(4), anova_table.round(4)

## Assumption Diagnostics

In [ ]:
bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(ols_model.resid, ols_model.model.exog)
dw_stat = durbin_watson(ols_model.resid)
vif_table = pd.DataFrame({
    'term': X.columns,
    'vif': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
})

diagnostic_tests = pd.DataFrame([
    {'test': 'Breusch-Pagan LM', 'statistic': bp_lm, 'p_value': bp_lm_p},
    {'test': 'Breusch-Pagan F', 'statistic': bp_f, 'p_value': bp_f_p},
    {'test': 'Durbin-Watson', 'statistic': dw_stat, 'p_value': np.nan},
])

diagnostic_tests.round(4), vif_table.round(4)